### CRISP-DM Phase 4.2 - Modeling : Continent-level correlation

In [ ]:
import pandas as pd
from pycountry_convert import country_alpha3_to_country_alpha2
from pycountry_convert import country_alpha2_to_continent_code
import ast
from scipy.stats import spearmanr
import matplotlib.pyplot as plt

In [ ]:
# Load the datasets
sensor = pd.read_csv('data/sensor.csv')
law = pd.read_csv('../law/data/law_classified.csv')

Aggregation per continent

In [ ]:
continent_names = {'AF': 'Africa', 'AS': 'Asia', 'EU': 'Europe', 'NA': 'North America', 
                   'OC': 'Oceania', 'SA': 'South America'}

fix_dict = {'TL': 'AS', 'AQ': 'AQ', 'TF': 'AQ'}
def country_to_continent(country):
    if country=='EUR' or country=='EUE' or country=='XKX':
        return 'EU'   
    else:
        country = country_alpha3_to_country_alpha2(country)
        if country in fix_dict:
            return fix_dict[country]
        else:
            continent = country_alpha2_to_continent_code(country)
            return continent
    
law['Country'] = law['Country'].apply(country_to_continent)
law.rename(columns={'Country': 'Continent'}, inplace=True)
sensor['Country'] = sensor['Country'].apply(country_to_continent)
sensor.rename(columns={'Country': 'Continent'}, inplace=True)

Compute relevant metrics

In [ ]:
## Legislative coverage
START = 2000
END = 2025
YEARS = list(range(START, END + 1))
HAZARDS = ['flood', 'drought', 'temperature_extremes', 'sea_level_rise', 'storm', 'wildfire',
          'melting', 'erosion', 'other', 'none']

law['Hazard'] = law['Hazard'].apply(lambda x: x if isinstance(x, list) else ast.literal_eval(x))
law_exploded = law.explode('Hazard').copy()

legislative_coverage = []

for continent, df in law_exploded.groupby('Continent', sort=True):
    for hazard in HAZARDS:
        subset = df[(df['Hazard'] == hazard) & (df['Year'] >= START) & (df['Year'] <= END)].sort_values('Year').copy()

        year_df = pd.DataFrame({'Year': YEARS})
        year_df['Continent'] = continent
        year_df['Continent_name'] = continent_names.get(continent)
        year_df['Hazard'] = hazard

        if subset.empty:
            year_df['Count'] = 0
        else:
            yearly_counts = subset.groupby('Year').size().reset_index(name='new_laws')
            year_df = year_df.merge(yearly_counts, on='Year', how='left')
            year_df['new_laws'] = year_df['new_laws'].fillna(0)
            year_df['Count'] = year_df['new_laws'].cumsum()

            # If last law appears before 2026
            last_year_with_law = subset['Year'].max()
            if last_year_with_law < END:
                final_count = year_df.loc[year_df['Year'] == last_year_with_law, 'Count'].values[0]
                year_df.loc[year_df['Year'] > last_year_with_law, 'Count'] = final_count

        legislative_coverage.append(year_df[['Continent', 'Continent_name', 'Year', 'Hazard', 'Count']])

legislative_coverage = pd.concat(legislative_coverage, ignore_index=True)
legislative_coverage.to_csv(f'../law/outputs/legislative_coverage_continent.csv', index=False)

In [ ]:
## Hazard intensity 
VARIABLES = ['2m_temperature', 'Instantaneous_wind_gust', 'Sea_level_anomaly', 
             'Snowmelt', 'SPEI', 'Total_precipitation']
BASELINE_END = 1999

sensor = sensor.groupby(['Continent', 'Year'])[VARIABLES].mean().reset_index()
sensor['Continent_name'] = sensor['Continent'].map(continent_names)
hazard_intensity = sensor.copy()

for var in VARIABLES:
    if var == 'Sea_level_anomaly' or var == 'Snowmelt':
        # SPEI (reference period 1991–2020) and SLA (reference period 1993-2012) already usable as is
        continue
    
    baseline = sensor[sensor['Year'] <= BASELINE_END].groupby('Continent')[var].mean().reset_index()
    baseline.columns = ['Continent', f'{var}_mean'] 
    hazard_intensity = hazard_intensity.merge(baseline, on='Continent', how='left')
    
    hazard_intensity[var] = ((hazard_intensity[var] - hazard_intensity[f'{var}_mean']))
    
    hazard_intensity = hazard_intensity.drop(columns=[f'{var}_mean'])
    
hazard_intensity = hazard_intensity[(hazard_intensity['Year'] >= 1995) & (hazard_intensity['Year'] <= 2025)].copy()
hazard_intensity = hazard_intensity[['Continent', 'Continent_name', 'Year', '2m_temperature', 
                                     'Instantaneous_wind_gust', 'Sea_level_anomaly', 'Snowmelt', 
                                     'SPEI', 'Total_precipitation']].copy()
hazard_intensity.to_csv(f'outputs/hazard_intensity_continent.csv', index=False)

Correlation

In [ ]:
hazard_variable_dict = {'flood': 'Total_precipitation', 'drought': 'SPEI', 
                        'temperature_extremes': '2m_temperature', 'sea_level_rise': 'Sea_level_anomaly', 
                        'storm': 'Instantaneous_wind_gust', 'melting': 'Snowmelt'}

sensor_continents = set(hazard_intensity['Continent'].dropna().unique())
law_continents = set(legislative_coverage['Continent'].dropna().unique())
common_continents = sensor_continents & law_continents

correlation = []

for hazard, variable in hazard_variable_dict.items():
    for continent in common_continents:
        continent_intensity = hazard_intensity[hazard_intensity['Continent'] == continent]
        continent_coverage = legislative_coverage[(legislative_coverage['Continent'] == continent) & (legislative_coverage['Hazard'] == hazard)]

        data = pd.merge(continent_intensity[['Year', variable]], continent_coverage[['Year', 'Count']], 
                        on='Year', how='inner').dropna()

        if data[variable].nunique() == 1 or data["Count"].nunique() == 1:  
            continue          
        try:
            rho, p = spearmanr(data[variable], data['Count'])
            correlation.append({'Hazard': hazard, 'Continent': continent, 
                                'Continent_name': continent_names.get(continent), 
                                'Rho': rho, 'p_value': p})
        except Exception as e:
            continue

correlation = pd.DataFrame(correlation)
correlation.to_csv(f'outputs/correlation_continent.csv', index=False)

Legislation lag

In [ ]:
lag_results = []

for lag in [0, 1, 2, 3, 4, 5]:
    lagged_coverage = legislative_coverage.copy()
    lagged_coverage['Year'] = lagged_coverage['Year'] - lag

    lagged_correlation = []

    for hazard, variable in hazard_variable_dict.items():
        for continent in common_continents:
            continent_intensity = hazard_intensity[hazard_intensity['Continent'] == continent]
            continent_coverage = lagged_coverage[(lagged_coverage['Continent'] == continent) & (lagged_coverage['Hazard'] == hazard)]

            data = pd.merge(continent_intensity[['Year', variable]], continent_coverage[['Year', 'Count']],
                            on='Year', how='inner').dropna()

            if data[variable].nunique() == 1 or data['Count'].nunique() == 1:
                continue
            try:
                rho, p = spearmanr(data[variable], data['Count'])
                lagged_correlation.append({'Hazard': hazard, 'Continent': continent, 'Rho': rho, 'p_value': p})
            except Exception as e:
                continue

    lagged_correlation = pd.DataFrame(lagged_correlation)
    sig_count = (lagged_correlation['p_value'] < 0.05).sum() if len(lagged_correlation) else 0
    total = len(lagged_correlation)

    lag_results.append({'lag': lag, 'significant': sig_count, 'total': total,
                        'pct': sig_count/total*100 if total > 0 else 0})

lag_results = pd.DataFrame(lag_results)
print(lag_results)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(lag_results['lag'], lag_results['pct'], marker='o', color='steelblue', linewidth=2)
ax.set_xlabel('Years lagged')
ax.set_ylabel('Percentage of significant correlations')
ax.set_title('Effect of lag on correlation significance')
ax.set_xticks(lag_results['lag'])
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('outputs/4_legislation_lag_continent.png', dpi=150, bbox_inches='tight')
plt.close()